
# Dimensionality reduction and clustering of Cell Painting profiles

**Dataset.** EU-OPENSCREEN bioactives, `IMTM_HepG2` subset ([Wolff et al.
2024](https://doi.org/10.1101/2024.08.27.609964)) — 2,459 bioactive compounds imaged in
HepG2 cells at the IMTM site, built into an AnnData by
[`scverse/cell-painting-io`](https://github.com/scverse/cell-painting-io/blob/cpjump1-anndata-notebook/eu_os_bioactives_to_anndata.ipynb).
**10,668 wells x 2,776 features**: 7 library plates (`B1001`-`B1007`) x 4 replicate
plates (`R1`-`R4`) x 384 wells.

**The substitution that makes scverse work here.** A *well* is an observation and a
*CellProfiler feature* is a variable. Everything downstream of PCA — neighbour graphs,
UMAP, Leiden, marker detection — is agnostic to what the columns mean, so it transfers
unchanged. What does *not* transfer is everything upstream: counts, library size,
log-transforms, and the mean-variance relationship that highly-variable-gene selection
is built on.

This notebook walks the standard
[single-cell best practices](https://www.sc-best-practices.org/) dimensionality-reduction
and clustering path, and at each step states the assumption being made, whether it
survives the move to morphological profiles, and what to do when it doesn't.

## 1. Setup

In [ ]:
import os
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

sc.settings.verbosity = 1
sc.set_figure_params(dpi=90, frameon=False, figsize=(4, 4))

# Override with EU_OS_H5AD if your copy lives elsewhere.
H5AD = Path(os.environ.get("EU_OS_H5AD", "../data/eu_os_imtm_hepg2.h5ad"))
CLIP = 10.0  # see the heavy-tail section of notebook 01

## 2. Load data and explore

In [ ]:
adata = ad.read_h5ad(H5AD)
adata

In [ ]:
# obs: one row per well. Plate/Replicate/Well locate it, EOS identifies the compound.
adata.obs.drop(columns=["smiles", "target_genes"]).head()

In [ ]:
print(adata.var.head(3), "\n")
print(pd.crosstab(adata.var.family, adata.var.compartment))
print("\nchannels:", adata.var.channel.value_counts(dropna=False).to_dict())


Two things to notice, both consequential.

**The stain has four channels, not five.** `DNA`, `ER`, `AGP`, `Mito` — there is no RNA
channel in this study. If you port code from a JUMP or CPJUMP1 notebook that iterates
over five channels, it will silently produce empty groups.

**The feature axis is wildly imbalanced.** `Texture` alone is 1,872 of 2,776 features
(67%), because texture is computed per channel x scale x angle. `ObjectSkeleton` is a
single feature. Any analysis that treats features as exchangeable — PCA, correlation
pruning, enrichment — is dominated by texture unless you intervene.


## 3. What the matrix already contains

> **Assumption (single-cell).** You start from raw integer counts and run QC ->
> size-factor normalization -> `log1p` -> HVG -> scale -> PCA.
>
> **Cell Painting.** The first three are already done, and done differently. `.X` here is
> a **per-plate robust z-score against DMSO**: for each replicate plate, subtract the
> DMSO median and divide by the DMSO MAD, feature by feature
> (`pycytominer`'s `mad_robustize`). The raw per-well medians are kept in
> `layers["aggregated"]`.
>
> **Consequence.** Re-running the transcriptomics preamble is not just unnecessary, it
> is wrong. `log1p` is undefined on negative values. There is no library size to
> normalize by — the control wells, not the total signal, are the reference. And
> `sc.pp.scale` would replace the per-plate DMSO reference with a global one, discarding
> the batch correction that is already baked in.

In [ ]:
X, raw = adata.X, adata.layers["aggregated"]
print(f".X            mean={X.mean():+.3f}  std={X.std():.2f}  min={X.min():.0f}  max={X.max():.0f}")
print(f"layers[aggr]  mean={raw.mean():+.1f}  std={raw.std():.1f}  (raw per-well medians, native units)")
print(f"non-finite in .X: {int((~np.isfinite(X)).sum())}")


| standard step | do it here? | why |
| --- | --- | --- |
| ambient / doublet removal | no | no droplets; the analogous artifacts are out-of-focus fields and mis-segmentation, handled upstream |
| size-factor normalization | **no** | no library size; DMSO controls are the reference |
| `log1p` | **no** | values are signed z-scores |
| HVG selection | replace | no mean-variance trend to fit — see section 5 |
| `sc.pp.scale` | **no** | already unit-MAD per plate; would destroy the per-plate reference |
| PCA / neighbours / UMAP / Leiden | **yes** | unchanged |


## 4. Heavy tails: the one Cell-Painting-specific fix you cannot skip

`mad_robustize` divides by the MAD of the DMSO wells *on that plate*. For a feature that
barely moves in DMSO, that denominator is tiny, and the quotient explodes. The result is
a matrix whose extreme values are three orders of magnitude beyond its bulk.

In [ ]:
q = np.percentile(adata.X, [0.1, 1, 50, 99, 99.9])
print(f"percentiles 0.1/1/50/99/99.9: {np.round(q, 2)}")
print(f"min {adata.X.min():.0f}   max {adata.X.max():.0f}")
print(f"share |x| > 10:  {(np.abs(adata.X) > 10).mean():.3%}")
print(f"share |x| > 100: {(np.abs(adata.X) > 100).mean():.4%}")


Fewer than 2% of values exceed 10 in absolute value, and 0.01% exceed 100 — but PCA
minimises squared error, so those few values decide the components. Run PCA both ways
and compare.

In [ ]:
from sklearn.decomposition import PCA


def pca_probe(matrix, label):
    p = PCA(n_components=20, svd_solver="randomized", random_state=0).fit(matrix)
    ev = p.explained_variance_ratio_
    top = np.argsort(np.abs(p.components_[0]))[::-1][:4]
    print(f"[{label:>12}] PC1={ev[0]:6.1%}  PC2={ev[1]:5.1%}  PC1-20={ev.sum():5.1%}")
    print(f"{'':15}PC1 driven by: {', '.join(adata.var_names[top])}")


pca_probe(adata.X.astype(np.float64), "unclipped")
pca_probe(np.clip(adata.X, -CLIP, CLIP).astype(np.float64), f"clip +/-{CLIP:.0f}")


Unclipped, **PC1 absorbs ~93% of the variance** and is driven by `ObjectSkeleton` and
`Neighbors` features — the ones whose DMSO MAD is near zero. That is not a phenotype, it
is a division artifact. After clipping, PC1 falls to ~31% and its top loadings are
`Texture` features on DNA and Mito, which is a plausible morphological axis.

> **What to do.** Clipping is the blunt fix and is what we use below. Alternatives, in
> rough order of how principled they are:
>
> - **Drop unstable features**: compute each feature's DMSO MAD per plate and remove
>   features whose MAD is in the bottom few percent on any plate. Attacks the cause
>   rather than the symptom. (The upstream notebook already dropped 174 features whose
>   MAD was exactly zero; near-zero ones remain.)
> - **Rank / quantile transform** each feature (`sklearn.preprocessing.QuantileTransformer`)
>   — fully outlier-immune, but discards effect magnitude, which you need for hit calling.
> - **Winsorize per feature** at its own 1st/99th percentile instead of a global cut.
> - **Robust PCA** or a Huber loss, if you want to keep the tails and still get stable
>   components.
>
> **TODO:** try one of these and see whether the confound audit in section 8 improves.

In [ ]:
adata.layers["robust_z"] = adata.X.copy()  # keep the unclipped values
adata.X = np.clip(adata.X, -CLIP, CLIP)
print(f"clipped to +/-{CLIP:.0f}; std now {adata.X.std():.2f}")


## 5. Feature selection: the HVG analogue

> **Assumption (single-cell).** HVG selection fits a mean-variance relationship — in
> counts, variance grows with mean — and keeps genes that are more variable than that
> trend predicts. It assumes most genes are uninformative noise.
>
> **Cell Painting.** There is no mean-variance trend to fit: every feature is already
> centred and scaled to the DMSO distribution, so all means are ~0 and all DMSO
> variances are ~1 by construction. A feature's variance across *all* wells therefore
> measures something different and more directly useful — how far compounds push it
> relative to control noise.
>
> **What replaces it.** Nothing, or `pycytominer.feature_select`. The field's standard is
> to prune *redundancy* rather than select *variability*, because texture features are
> near-duplicates of each other across scales and angles. `sc.pp.highly_variable_genes`
> will run on this matrix and return something, but its `seurat`/`cell_ranger` flavours
> bin by mean expression and are meaningless here.

In [ ]:
# Variance across all wells: informative here, but note that the top of this list is
# also where unstable features hide. Compare against each feature's DMSO variance.
is_dmso = (adata.obs.pert_type == "negcon").to_numpy()
var_all = adata.X.var(axis=0)
var_dmso = adata.X[is_dmso].var(axis=0)
ratio = var_all / np.maximum(var_dmso, 1e-6)

sel = pd.DataFrame(
    {"family": adata.var.family.to_numpy(), "var_all": var_all, "var_dmso": var_dmso, "ratio": ratio},
    index=adata.var_names,
)
print(sel.sort_values("ratio", ascending=False).head(8).round(2), "\n")
print("median variance ratio by family:")
print(sel.groupby("family", observed=True).ratio.median().sort_values(ascending=False).round(2))


A high `ratio` means a feature moves much more across the library than it does among
DMSO wells — the closest honest analogue of "highly variable".

> **TODO.** Decide whether to subset. Options: keep everything (PCA will handle the
> redundancy, at the cost of interpretability); keep the top *n* by `ratio`; or run
> `pycytominer.feature_select` with `correlation_threshold` to drop near-duplicate
> texture features. Redundancy matters most for notebook 02, where correlated features
> inside a family inflate enrichment scores.

## 5b. Should you regress out cell count?

Section 8 will show that `cell_count` correlates with PC1 at **r = +0.57**, which looks
like a textbook case for `sc.pp.regress_out`. It is not, and this section exists to show
why with numbers rather than an assertion.

> **Assumption.** `regress_out` fits an *independent* ordinary least squares model of
> each feature on the covariate, `feature ~ 1 + cell_count`, and replaces the feature by
> the residual. Three things are assumed: that the covariate's effect is **linear** on
> the feature's scale; that it is **additive**, i.e. the same offset for every compound
> with no interaction; and — the load-bearing one — that the covariate is a
> **nuisance**, so variation aligned with it carries nothing you want to keep.
>
> **In scRNA-seq** the classic targets are total counts and percent-mitochondrial. The
> first is sequencing depth, the second is largely dissociation stress. Both are
> genuinely technical: two cells of the same type sequenced at different depths *should*
> be pulled together, so removing that axis is a net gain.
>
> **In Cell Painting** `cell_count` is not depth. It is the number of objects
> CellProfiler segmented in the well, and that is a *readout*: cytotoxic compounds kill
> cells and antimitotics arrest division, so both push it down. In this dataset the 80
> annotated tubulin-binder wells average **767 cells against 1,226 for everything
> else**. Low cell count is not noise contaminating the tubulin phenotype, it is *part
> of* the tubulin phenotype. Regressing it out asks the matrix to forget the most
> reliable thing it knows about antimitotics.
>
> **Why the step belongs here.** `regress_out` works feature by feature, so it has to
> run on the feature matrix — after feature selection, before PCA. Running it on an
> embedding instead would remove a different thing.

So we will not take it on faith either way: run it, keep the result in a **layer** so
`.X` survives untouched, and measure what it does to the one thing in this dataset that
resembles ground truth — the annotated tubulin binders.

In [ ]:
import time

from sklearn.metrics import roc_auc_score, silhouette_samples
from sklearn.preprocessing import normalize

FIGDIR = Path("../figures")
FIGDIR.mkdir(exist_ok=True)
sc.settings.figdir = FIGDIR

is_dmso = (adata.obs.pert_type == "negcon").to_numpy()
is_trt = (adata.obs.pert_type == "trt").to_numpy()
is_tub = adata.obs.tubulin_binder.to_numpy()
cell_count = adata.obs.cell_count.to_numpy(float)

print(f"cell_count: tubulin binders {cell_count[is_tub].mean():.0f} "
      f"vs everything else {cell_count[~is_tub].mean():.0f}")

# Keep .X intact: copy it into a clearly named layer and regress *that*, so both paths
# stay available for the rest of the notebook and for notebooks 02 and 03.
adata.layers["regressed_cell_count"] = adata.X.copy()

t0 = time.time()
sc.pp.regress_out(adata, "cell_count", layer="regressed_cell_count",
                  n_jobs=min(16, os.cpu_count() or 1))
print(f"regress_out on {adata.n_vars} features x {adata.n_obs} wells: "
      f"{time.time() - t0:.2f}s")

# We need a PCA of each matrix to compare them, so this runs section 6's PCA early.
# Section 6 explains the parameter choices; re-running it there is idempotent.
sc.pp.pca(adata, n_comps=50, zero_center=True, svd_solver="arpack")
sc.pp.pca(adata, n_comps=50, layer="regressed_cell_count", zero_center=True,
          svd_solver="arpack", key_added="X_pca_regressed")
print("embeddings:", list(adata.obsm))

> **The obvious check is vacuous.** "How much of `corr(PC1, cell_count) = +0.57`
> survives?" has a guaranteed answer: **exactly zero**. Every feature is made orthogonal
> to `cell_count` by construction, so every linear combination of features is too — all
> 50 PCs come out at r = 0.000, not just PC1. A metric that cannot fail is not evidence
> that the step worked. The cell below reports it anyway to make the point, then
> measures four things that *can* fail:
>
> - **AUROC for `tubulin_binder`**, scoring each well by cosine similarity to the
>   tubulin consensus computed *without that well*. Leave-one-out stops a tubulin well
>   being scored against itself, and cosine makes it insensitive to the overall shrinkage
>   in scale that regressing out necessarily causes. Computed in PCA space and in
>   feature space.
> - **E-distance of tubulin binders from DMSO**, the effect size notebook 03 uses.
> - **decoupler feature-family z-scores**, the same quantity notebook 02 reports, so the
>   two notebooks stay comparable.
> - **percent replicating** against the position-matched null of notebook 03 — a pure
>   reproducibility measure that does not involve the tubulin annotation at all.
>
> The first three ask "can we still see the biology?"; the last asks "is the matrix less
> noisy?". They do not have to agree, and here they do not.

In [ ]:
from scipy.spatial.distance import cdist


def energy_distance(A, B):
    """E-distance between two clouds: 2*E|a-b| - E|a-a'| - E|b-b'|, squared euclidean."""
    return (2 * cdist(A, B, "sqeuclidean").mean()
            - cdist(A, A, "sqeuclidean").mean()
            - cdist(B, B, "sqeuclidean").mean())


def tubulin_auroc(M):
    """AUROC for tubulin_binder from leave-one-out cosine similarity to the consensus."""
    Mn = normalize(np.asarray(M))
    n = is_tub.sum()
    centroid = np.repeat(Mn[is_tub].sum(0)[None, :], len(Mn), axis=0)
    centroid[is_tub] -= Mn[is_tub]                                    # leave-one-out
    centroid = normalize(centroid / np.where(is_tub, n - 1, n)[:, None])
    score = (Mn * centroid).sum(1)
    keep = is_trt | is_dmso | is_tub
    return roc_auc_score(is_tub[keep], score[keep])


def tubulin_silhouette(P, n_dims=50):
    keep = is_trt | is_tub
    lab = is_tub[keep].astype(int)
    return silhouette_samples(P[keep][:, :n_dims], lab)[lab == 1].mean()


def percent_replicating(P, n_null=2500, seed=0):
    """Notebook 03, section 5: are a compound's 4 replicates more similar to each other
    than four *different* compounds sharing a well coordinate across library plates?
    The position-matched null is the strict one -- see notebook 03 for why.

    Notebook 03 reports ~1 point lower for the same quantity: it resamples the null from a
    different point in the RNG stream. Only the difference between conditions matters."""
    Pn = normalize(P[is_trt])
    eos = adata.obs.EOS.astype(str).to_numpy()[is_trt]
    well = adata.obs.Well.astype(str).to_numpy()[is_trt]
    plate = adata.obs.Plate.astype(str).to_numpy()[is_trt]
    rng = np.random.default_rng(seed)
    upper = np.triu_indices(4, k=1)

    same = []
    for _, pos in pd.Series(np.arange(len(eos)), index=eos).groupby(level=0):
        idx = pos.to_numpy()
        if len(idx) == 4:
            same.append((Pn[idx] @ Pn[idx].T)[upper].mean())

    null = []
    for _ in range(n_null):          # same well coordinate, four different library plates
        cand = np.flatnonzero(well == rng.choice(np.unique(well)))
        seen, pick = set(), []
        for i in rng.permutation(cand):
            if plate[i] not in seen:
                seen.add(plate[i])
                pick.append(i)
            if len(pick) == 4:
                break
        if len(pick) == 4:
            null.append((Pn[pick] @ Pn[pick].T)[upper].mean())

    same = np.asarray(same)
    return float((same > np.percentile(null, 95)).mean()), float(same.mean())


def family_zscores(M):
    """decoupler feature-family scores (notebook 02), as mean(tubulin) - mean(DMSO)."""
    import decoupler as dc

    from cellpainting_scverse import feature_sets

    tmp = ad.AnnData(X=np.ascontiguousarray(M), obs=adata.obs[[]].copy(),
                     var=adata.var.copy())
    net = feature_sets(tmp, keys=("compartment", "family", "channel"), min_size=5)
    pruned = dc.pp.prune(features=tmp.var_names.to_numpy(), net=net, tmin=5)
    dc.mt.zscore(tmp, pruned, tmin=5)
    scored = dc.pp.get_obsm(tmp, "score_zscore")
    z = pd.DataFrame(scored.X, index=tmp.obs_names, columns=scored.var_names)
    return z[is_tub].mean() - z[is_dmso].mean()


rows, fam = {}, {}
for label, pca_key, uns_key, M in [
    ("clipped .X", "X_pca", "pca", adata.X),
    ("regressed", "X_pca_regressed", "X_pca_regressed", adata.layers["regressed_cell_count"]),
]:
    P = adata.obsm[pca_key]
    pr, mean_same = percent_replicating(P)
    fam[label] = family_zscores(M)
    rows[label] = {
        "max |r| vs cell_count, 50 PCs": np.abs(np.corrcoef(P.T, cell_count)[-1, :-1]).max(),
        "PC1 variance ratio": adata.uns[uns_key]["variance_ratio"][0],
        "total feature variance": np.asarray(M).var(axis=0).sum(),
        "AUROC tubulin (PCA space)": tubulin_auroc(P),
        "AUROC tubulin (features)": tubulin_auroc(M),
        "silhouette, tubulin wells": tubulin_silhouette(P),
        "E-distance tubulin vs DMSO": energy_distance(P[is_tub], P[is_dmso]),
        "percent replicating": pr,
        "mean within-compound cosine": mean_same,
        "z channel:DNA (tub - DMSO)": fam[label]["channel:DNA"],
        "z channel:AGP (tub - DMSO)": fam[label]["channel:AGP"],
        "z channel:Mito (tub - DMSO)": fam[label]["channel:Mito"],
    }

comparison = pd.DataFrame(rows)
comparison["change"] = comparison["regressed"] - comparison["clipped .X"]
print(comparison.round(3).to_string())

In [ ]:
# Percent change in magnitude, so the direction of each effect is directly comparable.
shown = {
    "AUROC (PCA space)": "AUROC tubulin (PCA space)",
    "AUROC (feature space)": "AUROC tubulin (features)",
    "E-distance vs DMSO": "E-distance tubulin vs DMSO",
    "|z| channel:DNA": "z channel:DNA (tub - DMSO)",
    "percent replicating": "percent replicating",
}
rel = pd.Series({
    name: (abs(comparison.loc[key, "regressed"]) / abs(comparison.loc[key, "clipped .X"]) - 1) * 100
    for name, key in shown.items()
})

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].barh(range(len(rel)), rel.to_numpy(),
             color=["crimson" if v < 0 else "seagreen" for v in rel])
axes[0].set_yticks(range(len(rel)))
axes[0].set_yticklabels(rel.index, fontsize=9)
axes[0].axvline(0, color="k", lw=1)
axes[0].invert_yaxis()
axes[0].set_xlabel("change after regressing out cell_count (%)")
axes[0].set_title("Tubulin-binder detectability, and reproducibility", fontsize=10)

before, after = fam["clipped .X"], fam["regressed"]
axes[1].scatter(before, after, s=20, color="0.35")
lim = [min(before.min(), after.min()) - 0.7, max(before.max(), after.max()) + 0.7]
axes[1].plot(lim, lim, "k--", lw=1)
axes[1].set_xlim(lim)
axes[1].set_ylim(lim)
for key in before.index:
    if max(abs(before[key]), abs(after[key])) > 1.5 or abs(before[key] - after[key]) > 1.5:
        axes[1].annotate(key, (before[key], after[key]), fontsize=7,
                         xytext=(4, 3), textcoords="offset points")
axes[1].set_xlabel("family z-score, clipped .X")
axes[1].set_ylabel("family z-score, regressed")
axes[1].set_title("tubulin - DMSO, per feature family", fontsize=10)

for ax in axes:
    ax.set_frame_on(True)
fig.tight_layout()
fig.savefig(FIGDIR / "01_tubulin_regressout_before_after.png", dpi=200, bbox_inches="tight")
plt.show()

### Verdict: do not apply this to `.X`

The numbers, as printed above:

| | clipped `.X` | regressed | |
| --- | --- | --- | --- |
| max \|r\| with `cell_count` over 50 PCs | 0.569 | **0.000** | zero by construction, not a result |
| total feature variance | 16,455 | 14,129 | 14% of the matrix removed |
| AUROC tubulin, PCA space | 0.908 | **0.800** | worse |
| AUROC tubulin, feature space | 0.893 | **0.800** | worse |
| silhouette of tubulin wells | +0.004 | **−0.020** | worse, and now negative |
| E-distance tubulin vs DMSO | 33,379 | **20,408** | −39% |
| decoupler `channel:DNA` | −4.16 | −3.28 | signature 21% weaker |
| decoupler `channel:Mito` | +0.68 | **−4.96** | sign flip, an artefact |
| percent replicating, matched null | 65.7% | **69.5%** | *better* |

**On tubulin-binder detectability, regressing out `cell_count` makes things worse on
every measure.** Not marginally: AUROC drops 0.11 in both PCA and feature space, the
E-distance from DMSO falls by 39%, and the silhouette of the tubulin wells goes negative
— they are no longer even weakly separated from the rest of the library. The
`channel:DNA` depletion that notebook 02 identifies as the antimitotic signature loses a
fifth of its magnitude. This is the predicted failure: DNA-channel intensity and object
count are close to the same measurement, so removing one attenuates the other.

**The distortion is worse than simple attenuation.** `channel:Mito` moves from +0.68 to
−4.96 and becomes the single largest family score. Nothing biological happened. The
residual after removing a cell-count axis is not a cleaner version of the same profile,
it is a *different* profile, and it invites a mitochondrial story the data does not
support. If you regress out and then read feature families, this is the mistake you will
make.

**But it is not one-sided, and this is the interesting part.** Percent replicating against
the position-matched null *improves*, 65.7% → 69.5%. Both results are true and they are
consistent: `cell_count` is partly a shared technical axis, so removing it makes unrelated
wells less spuriously similar and replicate agreement looks better. It is also partly real
cytotoxic and antimitotic phenotype, and removing that costs you the hits you most want to
find. `regress_out` cannot tell the two apart, because a per-feature OLS has no way to know
which wells' low counts are meaningful.

> **Recommendation.** Leave `.X` clipped and unregressed.
> `layers['regressed_cell_count']` is kept so you can rerun any downstream step against it
> as a sensitivity analysis: if a hit survives both matrices, cell count is not the whole
> story; if it appears only in `.X`, check whether the compound is simply cytotoxic. That
> is a better use of the layer than picking one matrix and hoping.
>
> **Runtime.** 0.10 s here at `n_jobs=16`, ~0.2 s single-threaded. The standard warning
> that `regress_out` is slow comes from scRNA-seq, where it runs over 20,000+ genes; 2,776
> dense features against one covariate is a couple of matrix products. `n_jobs` is not the
> reason to hesitate.
>
> **If you do want a cell-count correction**, the assumption to fix is *additivity*, not
> the arithmetic. Options, in rough order of how much they respect the biology: include
> `cell_count` as a covariate in the differential test rather than editing the matrix; fit
> the regression on DMSO wells only and apply those coefficients everywhere, so the fit
> cannot be driven by the compounds you are trying to detect; or restrict the correction to
> features that are mechanically count-dependent (`Neighbors`, confluence-linked
> `AreaShape`) instead of all 2,776.


## 6. PCA

> **Assumption.** PCA wants roughly comparable feature scales and approximately
> symmetric distributions. Standard practice scales genes to unit variance first.
>
> **Cell Painting.** Already satisfied *by plate*, so pass `zero_center=True` but do not
> re-scale. The remaining violation is redundancy: with 1,872 texture features, the
> leading PCs describe texture covariance more than they describe biology.

In [ ]:
sc.pp.pca(adata, n_comps=50, zero_center=True, svd_solver="arpack")
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)
print("cumulative variance:", np.cumsum(adata.uns["pca"]["variance_ratio"])[[9, 19, 49]].round(3))


## 7. Neighbour graph and UMAP

These steps are entirely agnostic to what the features mean, so they transfer without
modification. The only Cell-Painting-specific choice is the metric: profiles are dense,
signed and continuous, so Euclidean distance in PC space is appropriate (unlike
correlation-based metrics often used on raw profiles).

In [ ]:
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=50)
sc.tl.umap(adata)


## 8. Confound audit — before you look at the embedding

This is the step with no real single-cell counterpart, and the one most worth your time.
In transcriptomics, batch is a nuisance you correct and move on. In a Cell Painting
screen there are three nuisances that look exactly like biology:

1. **Plate** — reagent age, incubation, imaging session.
2. **Well position** — edge evaporation, dispensing gradients. The 384 well positions
   are shared across every plate, so a position effect is *systematic*, not random.
3. **Cell count** — confluence and cytotoxicity change morphology globally. A toxic
   compound and a slow-growing one look alike.

Quantify each before interpreting anything.

In [ ]:
def eta_squared(adata, covariate, n_pcs=20):
    """Fraction of each PC's variance explained by a categorical covariate."""
    pcs = adata.obsm["X_pca"][:, :n_pcs]
    g = pd.Series(adata.obs[covariate].astype(str).to_numpy(), name="g")
    out = []
    for j in range(n_pcs):
        y = pcs[:, j]
        grand = y.mean()
        stats = pd.DataFrame({"y": y, "g": g}).groupby("g", observed=True)["y"].agg(["mean", "size"])
        ssb = (stats["size"] * (stats["mean"] - grand) ** 2).sum()
        out.append(ssb / ((y - grand) ** 2).sum())
    return np.asarray(out)


rows = {}
for cov in ["Plate", "Replicate", "Well", "pert_type"]:
    e = eta_squared(adata, cov)
    rows[cov] = {"PC1": e[0], "PC2": e[1], "mean_PC1_20": e.mean()}
audit = pd.DataFrame(rows).T.round(3)

r_count = np.corrcoef(adata.obsm["X_pca"][:, 0], adata.obs.cell_count)[0, 1]
print(audit, "\n")
print(f"corr(PC1, cell_count) = {r_count:+.3f}")


Read this table carefully — the ordering is the finding.

**Well position explains more variance than plate does** (mean eta^2 across PC1-20 of
roughly 0.16 versus 0.03). Per-plate normalization removed the plate offset but cannot
remove a gradient that recurs at the same coordinates on every plate.

**`cell_count` correlates with PC1 at about r = 0.57.** The single largest axis of
variation in this screen is substantially confluence and toxicity. Any cluster you find
must be checked against cell count before you call it a mechanism.

> **What to do.**
>
> - **Regress out cell count** (`sc.pp.regress_out(adata, ["cell_count"])`) if you want
>   mechanism-specific signal. It also removes genuine antiproliferative biology, so keep
>   both versions and compare.
> - **Keep it as a covariate** and report it alongside every cluster, rather than
>   removing it. Usually the more honest option.
> - **Treat position as spatial.** Well row/column are real 2D coordinates, so
>   `squidpy.gr.spatial_autocorr` on a plate graph tests for edge effects directly. Out
>   of scope here, but this is the right tool.
> - **Do not** reach for Harmony or scVI on `Plate` reflexively — the plate offset is
>   already gone, and what remains is positional.
>
> **TODO:** pick one, rerun sections 6-7, and see how the audit changes.

In [ ]:
adata.obs["log_cell_count"] = np.log10(adata.obs.cell_count.clip(lower=1))
sc.pl.umap(adata, color=["Plate", "Replicate", "pert_type", "log_cell_count"], ncols=2, size=6)

## 8b. Batch integration with Harmony: which covariate?

Section 8 leaves two candidate batch variables, `Plate` and `Replicate`, and the choice
between them is not a matter of taste — it depends on what each one actually indexes in
this experiment. Look at the design before running anything.

> **Assumption.** Harmony assumes the batch variable is **technical**: that the same
> biological populations are present in every batch, so it is safe to iteratively move
> each batch's cluster centroids onto a shared centroid. Under that assumption anything
> batch-specific is by definition an artefact.
>
> That assumption fails the moment a batch variable is **confounded with the biology**.
> If a batch contains compounds that no other batch contains, "remove what is specific to
> this batch" and "remove what is specific to these compounds" are the same instruction.

In [ ]:
# ---------------------------------------------------------------------------
# TODO (user decision). One variable controls everything below. The evidence is
# in this section; the recommendation is in the verdict at the end of it.
BATCH_KEY = "Plate"            # "Plate" | "Replicate" | None
# ---------------------------------------------------------------------------

trt_obs = adata.obs[is_trt]
per_compound = trt_obs.groupby("EOS", observed=True).agg(
    n_plates=("Plate", "nunique"), n_replicates=("Replicate", "nunique")
)

print("plates each compound appears on:")
print(per_compound.n_plates.value_counts().sort_index().to_string(), "\n")
print("replicates each compound appears in:")
print(per_compound.n_replicates.value_counts().sort_index().to_string(), "\n")
print("distinct compounds per Plate:")
print(trt_obs.groupby("Plate", observed=True).EOS.nunique().to_string(), "\n")
print("distinct compounds per Replicate:")
print(trt_obs.groupby("Replicate", observed=True).EOS.nunique().to_string(), "\n")
print(f"Plate x Replicate = {adata.obs.groupby(['Plate', 'Replicate'], observed=True).ngroups}"
      f" physical plates of ~{adata.n_obs // 28} wells")
print("\nmedian cell_count by Plate:")
print(adata.obs.groupby("Plate", observed=True).cell_count.median().to_string())
print("median cell_count by Replicate:")
print(adata.obs.groupby("Replicate", observed=True).cell_count.median().to_string())

Read that output before going further, because it settles the question.

**Every one of the 2,456 treated compounds appears on exactly one `Plate`**, and each
`Plate` carries ~350 compounds that no other plate carries. `Plate` is therefore not a
batch in Harmony's sense — it *is* the library layout, a design variable. Meanwhile 2,429
of 2,456 compounds appear in all four `Replicate`s, and each `Replicate` contains
essentially the whole library (~2,440 compounds). `Replicate` is the technical repeat:
same compounds, different physical plate, different day.

`Plate` and `Replicate` cross to give 28 physical plates of ~380 wells, so a "plate
effect" in the ordinary sense is split across both variables — the part that is shared by
all four copies of library plate B1005 lands on `Plate`, and the part shared by every
plate in replicate R2 lands on `Replicate`.

Median cell count varies more across `Plate` (1,128–1,550) than across `Replicate`
(1,259–1,319), which is easy to misread as evidence that `Plate` is the stronger nuisance.
It is the opposite: different plates hold different compounds, so some of that spread is
compounds killing cells.

> **Harmony is a *second* correction here.** `.X` is already per-plate `mad_robustize`
> (section 3), which centred and scaled every feature against the DMSO wells *on its own
> plate*. Additive and scale plate offsets are largely gone; what Harmony can still reach
> is higher-order structure that per-feature centring cannot. Expect a small effect, and
> be suspicious if it is large.
>
> **A version gotcha, since this is a tutorial.** With scanpy 1.12.4 and harmonypy 2.0.0,
> `sc.external.pp.harmony_integrate` raises a shape error: it transposes `Z_corr`, which
> harmonypy 2.0 already returns as `n_obs x n_pcs`. Calling harmonypy directly is a
> two-line workaround and is what the cell below does. Check whether your versions still
> need it.

In [ ]:
import importlib.metadata as ilmd

import harmonypy

print("scanpy", ilmd.version("scanpy"), "| harmonypy", ilmd.version("harmonypy"))


def harmony_integrate(adata, key, basis="X_pca", adjusted_basis="X_pca_harmony", **kwargs):
    """sc.external.pp.harmony_integrate equivalent, working around the scanpy 1.12 /
    harmonypy 2.0 transpose mismatch described above."""
    out = harmonypy.run_harmony(adata.obsm[basis], adata.obs, [key], **kwargs)
    adata.obsm[adjusted_basis] = np.ascontiguousarray(out.Z_corr, dtype="float32")
    return out


# Both, from the same PCA, so the comparison is apples to apples.
for key, basis in [("Replicate", "X_pca_harmony_replicate"), ("Plate", "X_pca_harmony_plate")]:
    t0 = time.time()
    out = harmony_integrate(adata, key, adjusted_basis=basis, max_iter_harmony=20, verbose=False)
    print(f"harmony({key:<9}) -> obsm['{basis}']  "
          f"{len(out.objective_harmony) - 1} iterations, {time.time() - t0:.1f}s")

In [ ]:
def eta_squared_rep(P, covariate, n_dims=20):
    """Section 8's eta_squared, generalized to any embedding."""
    g = pd.Series(adata.obs[covariate].astype(str).to_numpy(), name="g")
    out = []
    for j in range(n_dims):
        y = P[:, j]
        grand = y.mean()
        stats = pd.DataFrame({"y": y, "g": g}).groupby("g", observed=True)["y"].agg(["mean", "size"])
        out.append((stats["size"] * (stats["mean"] - grand) ** 2).sum() / ((y - grand) ** 2).sum())
    return np.asarray(out)


def eos_enrichment(P):
    """Neighbour enrichment (section 10) for compound identity, on an arbitrary rep."""
    probe = ad.AnnData(X=np.zeros((adata.n_obs, 1), "float32"), obs=adata.obs[["EOS", "Plate"]].copy())
    probe.obsm["rep"] = np.ascontiguousarray(P)
    sc.pp.neighbors(probe, n_neighbors=15, use_rep="rep")
    graph = probe.obsp["connectivities"].tocoo()
    out = {}
    for key in ("EOS", "Plate"):
        labels = probe.obs[key].astype(str).to_numpy()
        observed = float((labels[graph.row] == labels[graph.col]).mean())
        chance = float((probe.obs[key].value_counts(normalize=True).to_numpy() ** 2).sum())
        out[key] = observed / chance
    return out


REPS = {"X_pca": "no correction",
        "X_pca_harmony_replicate": "harmony(Replicate)",
        "X_pca_harmony_plate": "harmony(Plate)"}

harmony_rows = {}
for key, label in REPS.items():
    P = adata.obsm[key]
    pr, mean_same = percent_replicating(P)
    enr = eos_enrichment(P)
    harmony_rows[label] = {
        "eta2 Replicate (mean PC1-20)": eta_squared_rep(P, "Replicate").mean(),
        "eta2 Plate (mean PC1-20)": eta_squared_rep(P, "Plate").mean(),
        "eta2 Well (mean PC1-20)": eta_squared_rep(P, "Well").mean(),
        "AUROC tubulin": tubulin_auroc(P),
        "silhouette, tubulin wells": tubulin_silhouette(P),
        "neighbour enrichment, EOS": enr["EOS"],
        "neighbour enrichment, Plate": enr["Plate"],
        "mean within-compound cosine": mean_same,
        "percent replicating (matched null)": pr,
    }

harmony_compare = pd.DataFrame(harmony_rows)
print(harmony_compare.round(3).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

eta_keys = ["eta2 Replicate (mean PC1-20)", "eta2 Plate (mean PC1-20)", "eta2 Well (mean PC1-20)"]
sub = harmony_compare.loc[eta_keys]
x = np.arange(len(eta_keys))
for i, (label, col) in enumerate(sub.items()):
    axes[0].bar(x + (i - 1) * 0.26, col.to_numpy(), width=0.26, label=label)
axes[0].set_xticks(x)
axes[0].set_xticklabels(["Replicate", "Plate", "Well"])
axes[0].set_ylabel("variance explained (mean eta$^2$, PC1-20)")
axes[0].set_title("What each correction removes", fontsize=10)
axes[0].legend(fontsize=8, frameon=False)

sig_keys = ["percent replicating (matched null)", "neighbour enrichment, EOS", "AUROC tubulin"]
scale = {"percent replicating (matched null)": 1.0, "neighbour enrichment, EOS": 1 / 20,
         "AUROC tubulin": 1.0}
sub2 = harmony_compare.loc[sig_keys]
x = np.arange(len(sig_keys))
for i, (label, col) in enumerate(sub2.items()):
    vals = [col[k] * scale[k] for k in sig_keys]
    axes[1].bar(x + (i - 1) * 0.26, vals, width=0.26, label=label)
axes[1].set_xticks(x)
axes[1].set_xticklabels(["% replicating\n(matched null)", "EOS neighbour\nenrichment /20",
                         "AUROC\ntubulin"], fontsize=8)
axes[1].set_ylabel("compound signal retained")
axes[1].set_title("What each correction costs", fontsize=10)
axes[1].legend(fontsize=8, frameon=False)

for ax in axes:
    ax.set_frame_on(True)
fig.tight_layout()
fig.savefig(FIGDIR / "01_harmony_plate_vs_replicate.png", dpi=200, bbox_inches="tight")
plt.show()

### Verdict: `Replicate` is the defensible batch key, and `BATCH_KEY` is yours to set

As printed above:

| | no correction | harmony(`Replicate`) | harmony(`Plate`) |
| --- | --- | --- | --- |
| eta² `Replicate`, mean PC1–20 | 0.004 | **0.001** | 0.004 |
| eta² `Plate`, mean PC1–20 | 0.030 | 0.029 | **0.003** |
| eta² `Well`, mean PC1–20 | 0.156 | 0.158 | 0.163 |
| AUROC tubulin | 0.908 | 0.908 | 0.895 |
| silhouette, tubulin wells | +0.004 | +0.005 | +0.017 |
| neighbour enrichment, `EOS` | 14.7× | 14.8× | 13.3× |
| neighbour enrichment, `Plate` | 2.68× | 2.57× | 1.55× |
| mean within-compound cosine | 0.607 | 0.614 | 0.587 |
| percent replicating, matched null | 65.7% | **66.0%** | **60.1%** |
| iterations to converge | — | 1 | 5 |

**Does harmonizing `Plate` destroy compound signal?** Not destroy — erode, measurably and
in one direction. It does exactly what it is asked to do, dropping plate eta² from 0.030
to 0.003 and plate neighbour enrichment from 2.68× to 1.55×. The cost is that percent
replicating against the position-matched null falls **65.7% → 60.1%**, a loss of 5.6
points, or roughly one previously-reproducible compound in twelve. Compound-identity
neighbour enrichment falls 14.7× → 13.3×, mean within-compound cosine similarity 0.607 →
0.587, and tubulin AUROC 0.908 → 0.895. Nothing collapses; the library is still there. But
the direction is unambiguous and it follows from the design: because each compound lives on
exactly one plate, some genuine compound-set variation is inseparable from plate variation,
and Harmony removes it along with the artefact.

One caveat on how you grade this. Notebook 03 reports percent replicating against three
nulls, including a *plate-matched* one, and under harmony(`Plate`) that version improves —
but only because the same-plate null itself collapses once the plate effect is gone. Using
a plate-matched null to score a plate correction is circular. The well-matched null in the
table above is the one to read.

**harmony(`Replicate`) is close to a no-op, and slightly beneficial.** It converges in a
single iteration, because per-plate `mad_robustize` already removed most of what it
targets: replicate eta² was 0.004 to begin with and ends at 0.001. Percent replicating,
compound enrichment and within-compound similarity all nudge up; tubulin AUROC is unchanged
to three decimals. It costs 0.3 s and takes nothing away.

**The one honest surprise: neither correction touches the real problem.** `Well` position
explains eta² = 0.156, five times more than `Plate` and forty times more than `Replicate`,
and both runs leave it alone — 0.156 → 0.158 and → 0.163. `Well` is a 384-level covariate
confounded with compound identity in exactly the way `Plate` is, so Harmony is the wrong
instrument for it. The right fix is upstream: a plate-layout-aware normalization, or a
spatial-effect correction on the raw well-level profiles. If you take one thing from this
section, take that the dominant confound in this dataset is not the one Harmony is being
pointed at.

> **TODO — your call.** `BATCH_KEY` is set to **`"Plate"`** at the top of this section
> because that is what you asked for, and this notebook will not silently substitute
> something else. My recommendation, on the evidence above, is **`"Replicate"`**: `Plate` is
> a design variable rather than a batch, harmonizing it costs 5.6 points of percent
> replicating, and `Replicate` is the covariate that actually satisfies Harmony's "same
> biology in every batch" assumption. `BATCH_KEY = None` is also legitimate —
> `mad_robustize` has already done the heavy lifting and neither correction changes the
> tubulin result much.
>
> Whatever you set is recorded in `adata.uns['integration']`, and notebooks 02 and 03 read
> the embedding from there, so the consequence follows through the whole series. Notebook
> 03, section 5 recomputes percent replicating on your choice — that is the number to watch.

In [ ]:
# Everything downstream of here follows BATCH_KEY, set at the top of this section.
REP = {"Plate": "X_pca_harmony_plate",
       "Replicate": "X_pca_harmony_replicate",
       None: "X_pca"}[BATCH_KEY]
print(f"BATCH_KEY = {BATCH_KEY!r}  ->  using obsm['{REP}'] for the saved clustering")

# A second neighbour graph and UMAP on the chosen representation, kept alongside the
# uncorrected ones from section 7 rather than replacing them.
sc.pp.neighbors(adata, n_neighbors=15, use_rep=REP, key_added="harmony")
sc.tl.umap(adata, neighbors_key="harmony", key_added="X_umap_harmony")
sc.tl.leiden(adata, resolution=0.5, neighbors_key="harmony", key_added="leiden_harmony",
             flavor="igraph", n_iterations=2)

print(f"\nleiden_harmony: {adata.obs.leiden_harmony.nunique()} clusters")
print(pd.crosstab(adata.obs.leiden_harmony, adata.obs.tubulin_binder).T.to_string())


## 9. Clustering

> **Assumption (single-cell).** Clusters approximate discrete cell types. The population
> is a mixture of a modest number of well-separated states, and nearly every cell
> belongs to one.
>
> **Cell Painting.** Badly violated, in a specific and predictable way. Perturbation
> response is *continuous* (a dose-dependent push away from control) and *mostly absent*
> — at a single 10 uM dose, most bioactives do nothing measurable. So expect one enormous
> "indistinguishable from DMSO" cluster plus a handful of small clusters of genuinely
> active compounds, not a partition into balanced types.
>
> **How to read a cluster.** Not as a cell type but as a **shared morphological state**.
> Chemically unrelated compounds landing in one cluster is the interesting signal — it is
> the basis of guilt-by-association MOA prediction. Wells of the *same* compound landing
> together is a reproducibility check, not a discovery.

In [ ]:
for res in (0.25, 0.5, 1.0):
    key = f"leiden_{res}"
    sc.tl.leiden(adata, resolution=res, key_added=key, flavor="igraph", n_iterations=2)
    sizes = adata.obs[key].value_counts()
    print(f"res={res}: {len(sizes)} clusters | largest {sizes.iloc[0]:>5} wells "
          f"({sizes.iloc[0] / adata.n_obs:.0%}) | singleton-ish (<20): {(sizes < 20).sum()}")

In [ ]:
LEIDEN = "leiden_0.5"  # TODO: pick a resolution once you have seen the sizes above
sc.pl.umap(
    adata, 
    color=[LEIDEN], 
    frameon=False, 
    size=6)


## 10. Is the structure real? Neighbour enrichment

A UMAP of mostly-inactive compounds is very easy to over-read. Before interpreting it,
measure whether neighbouring wells actually share labels more often than chance. This
diagnostic comes from the upstream `cell-painting-io` notebook and is worth keeping in
every Cell Painting analysis.

In [ ]:
def neighbour_enrichment(adata, keys):
    """Observed vs chance rate of neighbouring wells sharing a label."""
    graph = adata.obsp["connectivities"].tocoo()
    rows = []
    for key in keys:
        labels = adata.obs[key].to_numpy()
        observed = float((labels[graph.row] == labels[graph.col]).mean())
        baseline = float((adata.obs[key].value_counts(normalize=True).to_numpy() ** 2).sum())
        rows.append({"covariate": key, "observed": observed, "chance": baseline, "ratio": observed / baseline})
    return pd.DataFrame(rows).set_index("covariate").round(3)


neighbour_enrichment(adata, ["Plate", "Replicate", "Well", "pert_type", "EOS"])


Same-compound (`EOS`) wells are strongly enriched — around 15x chance — and `Plate` around
2.7x, so both biology and batch are visible in the graph. But `Well` comes out highest of
all at roughly 22x, which deserves care rather than alarm, because **the ratio is sensitive
to how many categories a label has.** `Well` has 384 values and a chance baseline of 0.003;
`EOS` has 2,459 values and a baseline of 0.006. Comparing ratios across labels with such
different cardinality is not apples to apples.

Look at the absolute rates instead, where the ordering flips: neighbouring wells share a
compound **8.5%** of the time and a well position **5.8%**. So compound identity is the
stronger effect, and well position is a real, substantial, but secondary one. Notebook 03
measures the same thing far more directly, and agrees: replicate wells of a compound have
mean similarity +0.61, while wells merely sharing a plate coordinate reach only +0.12.

Both numbers are small in absolute terms, and that is the honest headline. Most compounds
at a single 10 uM dose move morphology too little to pull apart at all — which is why
screens score compounds by replicate reproducibility and induction (notebook 03) rather
than by clustering.

> **What to take from this.** Do not accept "these compounds cluster, so they share a
> mechanism" without checking position and cell count. Rerun this diagnostic after
> regressing out `cell_count`, and watch which ratios move — that tells you which of your
> structure was confound.


## 11. Characterizing clusters

> **Assumption.** `sc.tl.rank_genes_groups` defaults to a t-test on log-normalised
> counts; the `wilcoxon` and `logreg` options make fewer distributional assumptions.
> "Log fold change" presumes positive values on a multiplicative scale.
>
> **Cell Painting.** Use `wilcoxon` — it only needs ranks, which is safe for signed
> z-scores with heavy tails. **Ignore `logfoldchanges`**: it is computed as a ratio of
> means that can straddle zero, so it is meaningless here. Use `scores` (the rank
> statistic) or a plain difference in medians instead.

In [ ]:
sc.tl.rank_genes_groups(adata, groupby=LEIDEN, method="wilcoxon", n_genes=15)

top = sc.get.rank_genes_groups_df(adata, group=None)
top = top.merge(adata.var[["compartment", "family", "channel"]], left_on="names", right_index=True)
print(top.groupby("group", observed=True).head(3)[
    ["group", "names", "scores", "pvals_adj", "family", "channel"]
].to_string(index=False))

In [ ]:

# Which feature families define each cluster? Far more legible than feature names.
fam = (
    top.groupby(["group", "family"], observed=True).size()
    .unstack(fill_value=0)
    .pipe(lambda d: d.div(d.sum(axis=1), axis=0))
)
print("fraction of each cluster's top features by family:")
print(fam.round(2).to_string())


> **Suggestion.** Counting family membership among top features is a crude summary. The
> principled version is to score the families directly, per well, which is what
> notebook 02 does with `decoupler`.


## 12. What are the clusters made of?

Three questions worth asking of any Cell Painting clustering, in increasing order of
how much they tell you.

In [ ]:
obs = adata.obs
print("1. Is a cluster just a confound? cell count and plate composition per cluster")
print(obs.groupby(LEIDEN, observed=True).agg(
    n=("EOS", "size"), median_cells=("cell_count", "median"),
    n_plates=("Plate", "nunique"), frac_dmso=("pert_type", lambda s: (s == "negcon").mean()),
).round(2).to_string())

In [ ]:
print("2. Do the 4 replicates of a compound co-cluster? (reproducibility)")
trt = obs[obs.pert_type == "trt"]
per_cmp = trt.groupby("EOS", observed=True)[LEIDEN].agg(n_clusters="nunique", n_wells="size")
per_cmp = per_cmp[per_cmp.n_wells == 4]
print(f"   compounds with all 4 replicates in one cluster: "
      f"{(per_cmp.n_clusters == 1).mean():.1%} of {len(per_cmp)}")

In [ ]:
print("3. Are annotated tubulin binders concentrated somewhere? (positive control)")
ct = pd.crosstab(obs[LEIDEN], obs.tubulin_binder)
if True in ct.columns:
    ct["enrichment"] = (ct[True] / ct.sum(axis=1)) / obs.tubulin_binder.mean()
    print(ct.sort_values("enrichment", ascending=False).head(6).round(2).to_string())


The 20 annotated tubulin binders are the closest thing to ground truth in this dataset.
If they do not concentrate in one or two clusters, revisit the confound audit before
trusting any of the other clusters. Note that the annotation is noisy — several EGFR
inhibitors (lapatinib, pelitinib, canertinib) carry the tubulin flag.

## 13. Saving the integrated object

Notebooks 02 and 03 should not repeat any of this. Write one object that carries every
matrix and embedding computed above, plus a record in `.uns` of the choices that produced
them, so a reader can tell from the object alone which matrix is which.

The point of keeping four matrices is that different downstream questions need different
ones. Notebook 02's decoupler scoring wants the clipped `.X`, because feature-family
z-scores are only interpretable on comparable per-feature scales. Anything that reports
raw morphology in original units needs `layers['aggregated']`. A tail-sensitivity check
needs `layers['robust_z']`. And `layers['regressed_cell_count']` is there for the
sensitivity analysis described in section 5b — not because it should be the default.

In [ ]:
OUT = H5AD.with_name("eu_os_imtm_hepg2_integrated.h5ad")

adata.uns["integration"] = {
    "source_file": H5AD.name,
    "clip": float(CLIP),
    "X_is": f"per-plate mad_robustize robust z-scores, clipped to +/-{CLIP:.0f}",
    "feature_selection": "none — all features kept; variance-ratio report only (section 5)",
    "regress_out": "cell_count, stored in layers['regressed_cell_count'] only; .X NOT regressed",
    "batch_key": "none" if BATCH_KEY is None else BATCH_KEY,
    "rep": REP,
    "clustering_rep": REP,
    "clustering_key": "leiden_harmony",
    "clustering_resolution": 0.5,
    "uncorrected_clustering_key": LEIDEN,
    "recommended_batch_key": "Replicate  # see section 8b verdict; BATCH_KEY overrides",
    "versions": {p: ilmd.version(p) for p in ("scanpy", "anndata", "harmonypy", "pertpy", "decoupler")},
}

adata.write_h5ad(OUT, compression="gzip")
print(f"wrote {OUT}  ({OUT.stat().st_size / 1e6:.0f} MB)\n")
# anndata 0.13.3 yields a spurious None when iterating .layers; the file has 4.
print("layers :", [k for k in adata.layers if k])
print("obsm   :", list(adata.obsm))
print("obs    :", [c for c in adata.obs.columns if c.startswith("leiden")])
print("uns    :", list(adata.uns["integration"]))

### The contract

`data/eu_os_imtm_hepg2_integrated.h5ad`, 10,668 wells x 2,776 features.

| slot | what it is | who uses it |
| --- | --- | --- |
| `.X` | per-plate `mad_robustize` robust z, **clipped to ±10** | notebook 02 (decoupler), notebook 03 (per-feature tests) |
| `layers['aggregated']` | pre-normalization well-mean CellProfiler values | anything needing original units |
| `layers['robust_z']` | robust z, **unclipped** | tail-sensitivity checks (section 4) |
| `layers['regressed_cell_count']` | clipped `.X` with `cell_count` regressed out per feature | sensitivity analysis only (section 5b) |
| `obsm['X_pca']` | 50 PCs of clipped `.X` | baseline, and the input to both Harmony runs |
| `obsm['X_pca_regressed']` | 50 PCs of the regressed layer | section 5b comparison |
| `obsm['X_pca_harmony_replicate']` | Harmony on `Replicate` | recommended embedding |
| `obsm['X_pca_harmony_plate']` | Harmony on `Plate` | requested embedding |
| `obsm['X_umap']` | UMAP of `X_pca` | sections 7–12 |
| `obsm['X_umap_harmony']` | UMAP of `uns['integration']['rep']` | section 8b onward |
| `obs['leiden_0.25/0.5/1.0']` | Leiden on the uncorrected graph | section 9 |
| `obs['leiden_harmony']` | Leiden on the chosen embedding, res 0.5 | saved clustering |
| `uns['integration']` | clip value, feature selection, regress-out status, `batch_key`, `rep`, package versions, source file | provenance |

Downstream notebooks read the embedding from `uns['integration']['rep']` rather than
hard-coding a key, so changing `BATCH_KEY` in section 8b propagates through the series.

In [ ]:
# Slide-deck figures: the ones that carry an argument. dpi=200, tight bounding box.
SAVE = dict(dpi=200, bbox_inches="tight")

# 1. Clipping is not cosmetic (section 4).
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), sharey=True)
for ax, (matrix, title) in zip(axes, [(adata.layers["robust_z"], "unclipped robust z"),
                                      (adata.X, f"clipped to +/-{CLIP:.0f}")]):
    ratio = PCA(n_components=20, svd_solver="randomized", random_state=0).fit(
        np.asarray(matrix)).explained_variance_ratio_
    ax.bar(np.arange(1, 21), ratio * 100, color="0.35")
    ax.set_title(f"{title}\nPC1 = {ratio[0] * 100:.0f}% of variance", fontsize=10)
    ax.set_xlabel("PC")
    ax.set_frame_on(True)
axes[0].set_ylabel("variance explained (%)")
fig.tight_layout()
fig.savefig(FIGDIR / "01_pca_variance_unclipped_vs_clipped.png", **SAVE)
plt.show()

# 2. The confound audit (section 8). Well position, not plate, is the dominant one.
axes = sc.pl.umap(adata, color=["Plate", "Replicate", "pert_type", "log_cell_count"],
                  ncols=2, size=6, show=False)
# Four UMAP panels of 10,668 points: dpi=150 keeps this under the 2 MB commit limit.
plt.gcf().savefig(FIGDIR / "01_umap_confounds.png", dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(5, 3.4))
audit_fig = pd.DataFrame({cov: eta_squared_rep(adata.obsm["X_pca"], cov)
                          for cov in ["Well", "Plate", "Replicate", "pert_type"]})
ax.bar(audit_fig.columns, audit_fig.mean().to_numpy(), color="0.35")
ax.set_ylabel("variance explained (mean eta$^2$, PC1-20)")
ax.set_title("Confound audit: well position dominates", fontsize=10)
ax.set_frame_on(True)
fig.savefig(FIGDIR / "01_confound_audit_eta2.png", **SAVE)
plt.show()

# 3. Clusters, uncorrected and after the chosen correction.
axes = sc.pl.umap(adata, color=[LEIDEN], size=6, show=False)
plt.gcf().savefig(FIGDIR / "01_umap_leiden.png", **SAVE)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for ax, (key, title) in zip(axes, [("X_umap", "UMAP of X_pca (uncorrected)"),
                                   ("X_umap_harmony", f"UMAP of {REP}")]):
    emb = adata.obsm[key]
    ax.scatter(emb[:, 0], emb[:, 1], s=2, c="0.8", linewidths=0)
    ax.scatter(emb[is_tub, 0], emb[is_tub, 1], s=14, c="crimson", label="tubulin binder")
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
axes[0].legend(fontsize=8, frameon=False, loc="best")
fig.tight_layout()
fig.savefig(FIGDIR / "01_umap_harmony_vs_pca.png", **SAVE)
plt.show()

print("figures in", FIGDIR.resolve())
for png in sorted(FIGDIR.glob("*.png")):
    print(f"  {png.name:<48} {png.stat().st_size / 1024:6.0f} KB")

## Key takeaways

1. **Everything from PCA onward transfers unchanged.** Everything before it does not:
   no `log1p`, no size factors, no `scale`, no HVG.
2. **Clip the heavy tails, or nothing else works.** Unclipped, PC1 is ~93% variance and
   describes a division artifact in `ObjectSkeleton`/`Neighbors` features rather than a
   phenotype.
3. **Well position outranks plate as a confound**, because per-plate normalization
   cannot remove a gradient that recurs at the same coordinates on every plate.
4. **Cell count drives PC1** (r ~ 0.57). Check every cluster against it before calling
   anything a mechanism.
5. **A cluster is a morphological state, not a cell type.** Expect one huge inactive
   cluster; the value is in chemically unrelated compounds sharing a small one.
6. **Quantify before visualising.** Neighbour enrichment tells you how much of the UMAP
   is real: same-compound wells are ~14x enriched, but still only ~8% of neighbours.

## Things to try

- Drop low-DMSO-MAD features instead of clipping, and rerun the confound audit.
- `sc.pp.regress_out` on `cell_count`; compare tubulin-binder enrichment before/after.
- Prune correlated texture features with `pycytominer.feature_select`.
- Cluster *compound consensus profiles* (median over 4 replicates) instead of wells —
  10,668 wells becomes 2,459 compounds and the inactive blob shrinks.
- Bring in a second EU-OS site (`FMP_HepG2`, `MEDINA_HepG2`, `USC_HepG2`) and ask whether
  clusters replicate across sites — the real test of a morphological state.

## References

- [Single-cell best practices](https://www.sc-best-practices.org/)
- Wolff et al. 2024, *EU-OS bioactives Cell Painting* — https://doi.org/10.1101/2024.08.27.609964
- [scverse/cell-painting-io](https://github.com/scverse/cell-painting-io) — the AnnData builder
- Chandrasekaran et al. 2021, *Image-based profiling for drug discovery* — https://doi.org/10.1038/s41573-020-00117-w